In [ ]:
import pandas as pd

estados = [
    ('AC', -9.97, -67.81),
    ('AL', -9.66, -35.74),
    ('AP', 0.03, -51.05),
    ('AM', -3.12, -60.02),
    ('BA', -12.97, -38.50),
    ('CE', -3.73, -38.52),
    ('DF', -15.79, -47.88),
    ('ES', -20.32, -40.34),
    ('GO', -16.68, -49.25),
    ('MA', -2.53, -44.30),
    ('MT', -15.60, -56.10),
    ('MS', -20.47, -54.62),
    ('MG', -19.92, -43.94),
    ('PA', -1.45, -48.49),
    ('PB', -7.12, -34.86),
    ('PR', -25.43, -49.27),
    ('PE', -8.05, -34.88),
    ('PI', -5.09, -42.80),
    ('RJ', -22.91, -43.17),
    ('RN', -5.79, -35.21),
    ('RS', -30.03, -51.23),
    ('RO', -8.76, -63.90),
    ('RR', 2.82, -60.67),
    ('SC', -27.59, -48.55),
    ('SP', -23.55, -46.63),
    ('SE', -10.91, -37.07),
    ('TO', -10.18, -48.33)
]

df_estados = pd.DataFrame(estados, columns=['UF','Latitude','Longitude'])

In [ ]:
import requests
import pandas as pd

dados = []

for _, estado in df_estados.iterrows():

    url = (
        "https://power.larc.nasa.gov/api/temporal/monthly/point"
        f"?parameters=PRECTOTCORR_SUM"
        f"&community=AG"
        f"&longitude={estado['Longitude']}"
        f"&latitude={estado['Latitude']}"
        f"&start=2000"
        f"&end=2025"
        f"&format=JSON"
    )

    resposta = requests.get(url).json()

    if 'properties' not in resposta:
        print("Erro para:", estado['UF'])
        print(resposta)
        continue

    chuva = resposta['properties']['parameter']['PRECTOTCORR_SUM']

    for data, valor in chuva.items():
        dados.append({
            'UF': estado['UF'],
            'Ano': data[:4],
            'Mes': data[4:6],
            'Chuva_mm': valor
        })

df_chuva = pd.DataFrame(dados)

print(df_chuva.head())

   UF   Ano Mes  Chuva_mm
0  AC  2000  01    242.54
1  AC  2000  02    253.76
2  AC  2000  03    266.90
3  AC  2000  04    192.86
4  AC  2000  05     50.07


In [ ]:
df_chuva.to_csv('chuva_estados_brasil.csv', index=False, sep=';')

In [ ]:
import pandas as pd
import sqlite3

df_chuva = pd.read_csv('chuva_estados_brasil.csv', sep=';')

conn = sqlite3.connect('clima_brasil.db')

df_chuva.to_sql('chuva_estados', conn, if_exists='replace', index=False)

9126

In [ ]:
# Consultar a tabela com SQL
consulta = """
SELECT *
FROM chuva_estados
LIMIT 12
"""

resultado = pd.read_sql(consulta, conn)
resultado

,UF,Ano,Mes,Chuva_mm
0,AC,2000,1,242.54
1,AC,2000,2,253.76
2,AC,2000,3,266.90
3,AC,2000,4,192.86
4,AC,2000,5,50.07
5,AC,2000,6,16.15
6,AC,2000,7,36.62
7,AC,2000,8,73.27
8,AC,2000,9,93.90
9,AC,2000,10,159.97


In [ ]:
# Chuva total por estado
consulta = """
SELECT
    UF,
    SUM(Chuva_mm) AS chuva_total
FROM chuva_estados
GROUP BY UF
ORDER BY chuva_total DESC
"""

pd.read_sql(consulta, conn)

,UF,chuva_total
0,PA,135191.94
1,AP,105306.68
2,AM,102463.96
3,MA,91717.36
4,RO,85258.48
5,AC,84831.90
6,PR,83212.78
7,SC,79583.80
8,RR,79380.26
9,TO,76822.92


In [ ]:
# Chuva média anual por estado
consulta = """
SELECT
    UF,
    Ano,
    AVG(Chuva_mm) AS chuva_media_anual
FROM chuva_estados
GROUP BY UF, Ano
ORDER BY UF, Ano
"""

df_anual = pd.read_sql(consulta, conn)
df_anual

,UF,Ano,chuva_media_anual
0,AC,2000,279.370769
1,AC,2001,296.303077
2,AC,2002,274.521538
3,AC,2003,292.952308
4,AC,2004,292.741538
...,...,...,...
697,TO,2021,240.532308
698,TO,2022,230.776923
699,TO,2023,186.047692
700,TO,2024,176.486154


In [ ]:
# Chuva média mensal por estado
consulta = """
SELECT
    UF,
    Mes,
    AVG(Chuva_mm) AS chuva_media_mensal
FROM chuva_estados
GROUP BY UF, Mes
ORDER BY UF, Mes
"""

df_mensal = pd.read_sql(consulta, conn)
df_mensal

,UF,Mes,chuva_media_mensal
0,AC,1,239.467692
1,AC,2,241.598462
2,AC,3,233.348462
3,AC,4,149.192308
4,AC,5,70.563846
...,...,...,...
346,TO,9,37.018077
347,TO,10,104.511538
348,TO,11,185.164231
349,TO,12,222.678077
